# Advanced Retrieval — COSORA (Sesión A + D)

**Sesión A:** cross-encoder reranker + query decomposition (exploración inicial).

**Sesión D** — ver flags `RUN_*` en Setup (sec. 0).

**Ruta rápida (por defecto):** D1 referencias + **D5** sweep pool 50/100.  
Sesiones A, D2, D3, D4 están **desactivadas** (ya evaluadas → resultado negativo).

**Prerequisitos:** ingestion (`e5`) + grafos JSON de `graph_rag.ipynb` sec 2.


## 0. Setup

In [1]:
%pip install -q chromadb sentence-transformers rank_bm25 transformers accelerate python-dotenv nltk pandas torch google-cloud-storage

import nltk
nltk.download('punkt', quiet=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 89.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 122.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 100.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

True

In [2]:
# ═══════════════════════════════════════════════════════════════════════
# Config + COSORA core
# ═══════════════════════════════════════════════════════════════════════
import json
import os
import re
import sys
from pathlib import Path
from typing import Callable

import chromadb
import numpy as np
import pandas as pd
from openai import OpenAI
from rank_bm25 import BM25Okapi

IN_COLAB = "google.colab" in sys.modules
RUNTIME = "colab" if IN_COLAB else "local"

# ─── Retrieval ────────────────────────────────────────────────────────
EMBED_BACKEND = "e5"
RETRIEVAL_K = 100
TOP_N = 10
RRF_K = 60
EVAL_K_LIST = [5, 10]
LLM_JUDGE_MODEL = "gpt-4o-mini"

# ─── Grafo ────────────────────────────────────────────────────────────
SUBSET_SIZE = 10
ROUTER_GRAPH_SOURCE = "original"
GRAPH_TOP_K = 5
GRAPH_MIN_COS = 0.80
MATCH_MIN_OVERLAP = 0.5

# ─── Reranker ─────────────────────────────────────────────────────────
RERANKER_MODEL = "BAAI/bge-reranker-base"           # Sesión A
RERANKER_MODEL_M3 = "BAAI/bge-reranker-v2-m3"       # D4 multilingüe
RERANK_CANDIDATES = 20              # Sesión A / D2 (referencia histórica)
RERANK_BATCH_SIZE = 16

# ─── Qué ejecutar (ahorra juicios LLM / tokens) ───────────────────────
RUN_D1 = True                 # baseline + router (requerido para D5)
RUN_D1_SPLIT = False          # factual/transversal: derivable de router_d
RUN_SESSION_A = False           # rerank always-on n=20 — DESCARTADO (Sesión D)
RUN_D2 = False                  # rerank condicional — DESCARTADO
RUN_D3 = False                  # decomposition — DESCARTADO
RUN_D4 = False                  # m3 multilingüe — DESCARTADO
RUN_D5 = True                   # sweep pool — HIPÓTESIS ACTIVA

# D5 — sweep pool (literatura: 50–100; n=20 ya medido en Sesión A)
D5_CANDIDATE_SWEEP = [50, 100]  # añade 20 si quieres replicar: [20, 50, 100]
D5_RECALL_K = 100
D5_RUN_M3_IF_IMPROVES = True      # m3 solo si mejor pool supera router

# Resultados archivados (Sesión D previa) — para sec. 10 sin re-ejecutar
ARCHIVED_NDCG5 = {
    "graph_router_rerank": 0.888,      # pool=20, always-on
    "router_cond_rerank": 0.918,
    "router_cond_rerank_m3": 0.909,
    "graph_router_rerank_m3": 0.900,
    "decompose_router": 0.898,           # 8 compuestas
}

# D2 — reranker condicional (umbrales sobre score RRF del top-5)
RERANK_SCORE_THRESHOLD = 0.025   # top1 por debajo → poca confianza
RERANK_MARGIN_THRESHOLD = 0.005  # top1−top5 estrecho → empate

# ─── Query decomposition ─────────────────────────────────────────────
DECOMPOSE_MODEL = "gpt-4o-mini"
MAX_SUB_QUERIES = 4

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

from dotenv import load_dotenv
if RUNTIME == "colab":
    load_dotenv("/content/drive/MyDrive/variablentorno/.env")
else:
    for p in [".env", "../.env", "../../.env"]:
        if Path(p).exists():
            load_dotenv(p)
            break

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


def resolve_paths(runtime, chroma_path_override=None):
    if runtime == "colab":
        docs_dir = "/content/drive/MyDrive/RAG_UPC_Final_project"
        chroma_path = chroma_path_override or f"{docs_dir}/chroma_db"
        graph_dir = f"{docs_dir}/graph"
    else:
        nb_dir = Path(".").resolve()
        if nb_dir.name in ("experiments", "notebooks"):
            root = nb_dir.parents[1] if nb_dir.name == "experiments" else nb_dir.parent
        else:
            root = nb_dir
        docs_dir = str(root / "data" / "raw")
        chroma_path = chroma_path_override or str(root / "data" / "chroma_db")
        graph_dir = str(root / "data" / "graph")
    Path(graph_dir).mkdir(parents=True, exist_ok=True)
    return docs_dir, chroma_path, graph_dir

DOCS_DIR, CHROMA_PATH, GRAPH_DIR = resolve_paths(RUNTIME)

EMBED_BACKENDS = {
    "e5": {
        "model_id": "intfloat/multilingual-e5-base",
        "collection": "cosora_chunks_e5",
        "query_prefix": "query: ",
        "doc_prefix": "passage: ",
    },
}

STOPWORDS_ES = {
    "de","la","que","el","en","y","a","los","del","se","las","por","un","para","con","no","una",
    "su","al","es","lo","como","más","pero","sus","le","ya","o","este","sí","porque","esta",
}

_NORMALIZE_RE = re.compile(r"[^\wáéíóúñü]+", re.IGNORECASE)


class Embedder:
    def __init__(self, backend="e5"):
        self.cfg = EMBED_BACKENDS[backend]
        self._st = None

    def _load(self):
        if self._st is None:
            from sentence_transformers import SentenceTransformer
            self._st = SentenceTransformer(self.cfg["model_id"])

    def embed_one(self, text, *, is_query=False):
        self._load()
        p = self.cfg["query_prefix"] if is_query else self.cfg["doc_prefix"]
        v = self._st.encode(p + text)
        return v.tolist() if hasattr(v, "tolist") else list(v)

    def embed_batch(self, texts, *, is_query=False, batch_size=32):
        self._load()
        p = self.cfg["query_prefix"] if is_query else self.cfg["doc_prefix"]
        return self._st.encode([p + t for t in texts], batch_size=batch_size, show_progress_bar=False).tolist()


def strip_doc_prefix(text, backend="e5"):
    p = EMBED_BACKENDS[backend]["doc_prefix"]
    return text[len(p):] if text.startswith(p) else text


def tokenize_bm25(text, stemmer=None):
    words = [w for w in re.sub(r"[^\w\s]", "", text.lower()).split() if w and w not in STOPWORDS_ES]
    if stemmer:
        return [stemmer.stem(w) for w in words]
    return words


def build_bm25_index(collection):
    try:
        from nltk.stem.snowball import SnowballStemmer
        stemmer = SnowballStemmer("spanish")
    except Exception:
        stemmer = None
    data = collection.get()
    docs, metas = data["documents"], data["metadatas"]
    return BM25Okapi([tokenize_bm25(d, stemmer) for d in docs]), docs, metas, stemmer


def dense_search(collection, embedder, query, k=100):
    q = embedder.embed_one(query, is_query=True)
    res = collection.query(query_embeddings=[q], n_results=k)
    return [{"text": d, "meta": m, "rank_dense": i} for i, (d, m) in enumerate(zip(res["documents"][0], res["metadatas"][0]))]


def bm25_search(query, bm25_index, docs, metas, stemmer, k=100):
    scores = bm25_index.get_scores(tokenize_bm25(query, stemmer))
    top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [{"text": docs[i], "meta": metas[i], "rank_bm25": r} for r, i in enumerate(top)]


def rrf_fusion_baseline(dense, bm25, rrf_k=60, top_n=10):
    scores = {}
    for item in dense:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0})
        scores[cid]["score"] += 1.0 / (rrf_k + item["rank_dense"])
    for item in bm25:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0})
        scores[cid]["score"] += 1.0 / (rrf_k + item["rank_bm25"])
    return sorted(scores.values(), key=lambda x: x["score"], reverse=True)[:top_n]


collection = chromadb.PersistentClient(path=CHROMA_PATH).get_collection("cosora_chunks_e5")
embedder = Embedder("e5")
bm25_index, all_docs, all_metas, stemmer = build_bm25_index(collection)

print(f"RUNTIME={RUNTIME}  chunks={collection.count()}")
print(f"Chroma: {CHROMA_PATH}")
print(f"Graph dir: {GRAPH_DIR}")
print("✅ Setup listo")


Mounted at /content/drive
RUNTIME=colab  chunks=582
Chroma: /content/drive/MyDrive/RAG_UPC_Final_project/chroma_db
Graph dir: /content/drive/MyDrive/RAG_UPC_Final_project/graph
✅ Setup listo


## 1. Grafo + router (`graph_router`)

Clasificador **alineado con `graph_rag.ipynb` sec. 11** (regex + LLM few-shot).


In [3]:
def graph_json_path(src: str) -> Path:
    tag = f"sub{SUBSET_SIZE}" if SUBSET_SIZE else "full"
    clus = "raw" if src == "original" else "cl"
    return Path(GRAPH_DIR) / f"graph_actas_e5_{src}_{tag}_bal_{clus}.json"


def load_graph_variant(src: str) -> dict:
    path = graph_json_path(src)
    if not path.exists():
        hits = sorted(Path(GRAPH_DIR).glob(f"graph_actas_e5_{src}_*.json"))
        if not hits:
            raise FileNotFoundError(f"No hay grafo '{src}' en {GRAPH_DIR}. Ejecuta graph_rag.ipynb sec 2.")
        path = hits[-1]
        print(f"⚠️  Usando fallback: {path.name}")
    with open(path, encoding="utf-8") as f:
        gd = json.load(f)
    rels = [tuple(r) for r in gd["relations"]]
    texts = [f"{s} {r} {o}" for s, r, o in rels]
    mat = np.array(embedder.embed_batch(texts, is_query=False, batch_size=32), dtype=np.float32)
    norm = mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12)
    print(f"[{src}] {len(rels)} triples desde {path.name}")
    return {"relations": rels, "texts": texts, "norm": norm, "meta": gd.get("meta", {})}


def _tokenize_match(s):
    return set(t for t in _NORMALIZE_RE.split(s.lower()) if t and len(t) > 2 and t not in STOPWORDS_ES)


def _match_wordset(triple, doc):
    s, _, o = triple
    dw = _tokenize_match(doc)
    if not dw:
        return False
    def ok(words):
        if not words:
            return False
        return len(words & dw) / len(words) >= MATCH_MIN_OVERLAP
    return ok(_tokenize_match(s)) and ok(_tokenize_match(o))


def chunks_matching_triple(triple, docs):
    return [i for i, d in enumerate(docs) if _match_wordset(triple, d)]


def retrieve_graph_variant(query, variant, k=GRAPH_TOP_K):
    q = np.array(embedder.embed_one(query, is_query=True), dtype=np.float32)
    q = q / (np.linalg.norm(q) + 1e-12)
    sims = variant["norm"] @ q
    top = np.argsort(-sims)[:k]
    return [{"triple": variant["relations"][i], "text": variant["texts"][i], "score": float(sims[i]), "rank_graph": rk} for rk, i in enumerate(top)]


def rrf_fusion_graph(dense, bm25, graph_triples, top_n=10, min_cos=GRAPH_MIN_COS):
    scores = {}
    for item in dense:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0})
        scores[cid]["score"] += 1.0 / (RRF_K + item["rank_dense"])
    for item in bm25:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0})
        scores[cid]["score"] += 1.0 / (RRF_K + item["rank_bm25"])
    for t in graph_triples:
        if t["score"] < min_cos:
            continue
        for i in chunks_matching_triple(t["triple"], all_docs):
            cid = all_metas[i]["chunk_id"]
            scores.setdefault(cid, {"text": all_docs[i], "meta": all_metas[i], "score": 0.0})
            scores[cid]["score"] += 1.0 / (RRF_K + t["rank_graph"])
    return sorted(scores.values(), key=lambda x: x["score"], reverse=True)[:top_n]


def retrieve_graph_rag_with(query, variant, top_n=TOP_N):
    d = dense_search(collection, embedder, query, k=RETRIEVAL_K)
    b = bm25_search(query, bm25_index, all_docs, all_metas, stemmer, k=RETRIEVAL_K)
    g = retrieve_graph_variant(query, variant, k=GRAPH_TOP_K)
    return rrf_fusion_graph(d, b, g, top_n=top_n)


def retrieve_baseline(query, top_n=TOP_N):
    d = dense_search(collection, embedder, query, k=RETRIEVAL_K)
    b = bm25_search(query, bm25_index, all_docs, all_metas, stemmer, k=RETRIEVAL_K)
    return rrf_fusion_baseline(d, b, rrf_k=RRF_K, top_n=top_n)


# ─── Clasificador (graph_rag sec 11) ─────────────────────────────────
TRANSVERSAL_TRIGGERS = [
    "todas", "todos", "todas las", "todos los",
    "agrupa", "agrupar", "clasifica", "clasificar",
    "compara", "comparar", "comparación",
    "frecuente", "frecuentes", "frecuencia",
    "lista", "listar", "enumera", "enumerar",
    "tipos de", "tipo de tareas", "qué tipo",
    "patrón", "patrones", "tendencia", "tendencias",
    "conjunto", "varias actas", "todas las actas",
    "general", "globalmente", "en general",
    "qué incidencias", "qué problemas", "qué elementos",
    "más comunes", "más frecuentes",
    "cuáles son las",
    "incumplimientos", "solicitudes realizadas",
]

FACTUAL_TRIGGERS = [
    "qué se decidió", "qué se acordó",
    "estado del", "estado de la",
    "cuál es", "cuáles es",
    "cuántos", "cuántas",
    "a qué altura", "a qué distancia",
    "modelo de", "función de", "sigla",
    "documentación", "certificados",
    "ar-29",
]


def _normalize_q(q: str) -> str:
    return q.lower().strip()


def classify_query_regex(query: str) -> str | None:
    q = _normalize_q(query)
    score_t = sum(1 for w in TRANSVERSAL_TRIGGERS if w in q)
    score_f = sum(1 for w in FACTUAL_TRIGGERS if w in q)
    if score_t >= 1 and score_t > score_f:
        return "transversal"
    if score_f >= 1 and score_f > score_t:
        return "factual"
    return None


_CLASSIFY_SYSTEM = (
    "Eres un clasificador binario de consultas sobre actas de obra ferroviaria. "
    "Tu única tarea: emitir una etiqueta entre 'factual' y 'transversal'.\n"
    "- factual: pregunta puntual sobre un dato concreto contenido en UN documento.\n"
    "- transversal: pregunta agregada que requiere sintetizar VARIOS documentos."
)

_CLASSIFY_FEW_SHOT = [
    ("¿Qué se decidió sobre el talud?", "factual"),
    ("¿Cuántos días hay para devolver el acta?", "factual"),
    ("¿Cuáles son las incidencias más frecuentes en todas las actas?", "transversal"),
    ("Lista todas las solicitudes a la constructora", "transversal"),
    ("¿Qué modelo de luminaria debe instalarse?", "factual"),
    ("Agrupa las incidencias por tipo", "transversal"),
]


def classify_query_llm(query: str) -> str:
    msgs = [{"role": "system", "content": _CLASSIFY_SYSTEM}]
    for q, lbl in _CLASSIFY_FEW_SHOT:
        msgs.append({"role": "user", "content": q})
        msgs.append({"role": "assistant", "content": lbl})
    msgs.append({"role": "user", "content": query})
    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini", messages=msgs, temperature=0, max_tokens=4,
        )
        out = (resp.choices[0].message.content or "").strip().lower()
        return "transversal" if "transversal" in out else "factual"
    except Exception:
        return "factual"


def classify_query(query: str) -> str:
    label = classify_query_regex(query)
    if label is not None:
        return label
    return classify_query_llm(query)


_router_variant = load_graph_variant(ROUTER_GRAPH_SOURCE)


def retrieve_router(query, top_n=TOP_N):
    if classify_query(query) == "transversal":
        return retrieve_graph_rag_with(query, _router_variant, top_n=top_n)
    return retrieve_baseline(query, top_n=top_n)


def retrieve_router_k(q, k):
    return retrieve_router(q, top_n=k)


def retrieve_baseline_k(q, k):
    return retrieve_baseline(q, top_n=k)


print("✅ baseline + graph_router listos")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

[original] 426 triples desde graph_actas_e5_original_sub10_bal_raw.json
✅ baseline + graph_router listos


## 2. Benchmark + evaluación (D1 — alineado con `graph_rag`)

Judge con criterios detallados + `eval_variant` con strip de prefijo y conteo de errores.


In [4]:
BENCHMARK_QUERIES_FACTUAL = [
    "¿Qué se decidió sobre el talud?",
    "¿Cuál es el estado del camino provisional?",
    "¿Qué incidencias AR-29 aparecen?",
    "¿Qué responsable está asignado a las acciones sobre el talud?",
    "¿Qué se acordó sobre hormigonado de zapatas?",
    "Estado de las instalaciones de megafonía",
    "¿Qué documentación debe aportar la UTE sobre estabilidad?",
    "¿Cuál es la función de la sigla DEO en un acta de visita de obra?",
    "¿Cuántos días hay de plazo para devolver el acta firmada?",
    "¿Qué modelo de luminaria debe instalarse en los báculos del andén?",
    "¿Qué certificados de material debe aportar la UTE sobre las hincas de perfiles?",
    "¿A qué altura respecto al suelo se van a instalar las unidades exteriores de climatización?",
]

BENCHMARK_QUERIES_TRANSVERSAL = [
    "¿Cuáles son las incidencias más frecuentes en todas las actas?",
    "¿Qué elementos constructivos presentan más incidencias?",
    "¿Qué problemas pueden afectar a la seguridad de la explotación ferroviaria?",
    "Agrupa todas las incidencias detectadas en las actas y clasifícalas por tipo",
    "¿Cuáles son las acciones pendientes más frecuentes en las actas?",
    "¿Cuáles son los problemas más frecuentes en instalaciones eléctricas y luminarias?",
    "Lista todas las solicitudes realizadas a la constructora",
    "¿Qué incumplimientos normativos se detectan en el conjunto de actas?",
    "¿Qué tipo de tareas suelen quedar sin resolver?",
]

BENCHMARK_QUERIES_COMPOSITE = [
    "¿Qué se decidió sobre el talud y qué acciones quedan pendientes?",
    "Compara las incidencias de luminarias con las de instalaciones eléctricas en las actas",
    "¿Qué documentación debe aportar la UTE y cuál es el estado de las instalaciones de megafonía?",
    "Lista las solicitudes a la constructora y agrupa las incidencias por tipo constructivo",
    "¿Qué problemas de seguridad ferroviaria aparecen y qué responsable está asignado al talud?",
]

# Extra sintéticas para D3
BENCHMARK_QUERIES_COMPOSITE_EXTRA = [
    "¿Cuántos días hay de plazo para devolver el acta y qué incidencias AR-29 aparecen?",
    "¿Qué modelo de luminaria debe instalarse y cuál es el estado del camino provisional?",
    "Enumera las solicitudes a la constructora y compara incidencias de talud con zapatas",
]

BENCHMARK_QUERIES_COMPOSITE_ALL = BENCHMARK_QUERIES_COMPOSITE + BENCHMARK_QUERIES_COMPOSITE_EXTRA
BENCHMARK_QUERIES = BENCHMARK_QUERIES_FACTUAL + BENCHMARK_QUERIES_TRANSVERSAL

JUDGE_DEBUG = False
_judge_debug_remaining = 3


def judge_relevance(query: str, chunk_text: str) -> int:
    global _judge_debug_remaining
    system = (
        "Eres un evaluador de relevancia para un sistema de búsqueda sobre actas de obra "
        "ferroviaria. Tu única tarea: emitir un dígito (0, 1 o 2) que valore si el FRAGMENTO "
        "es relevante para la PREGUNTA. Sin texto adicional."
    )
    user = (
        f"PREGUNTA: {query}\n\n"
        f"FRAGMENTO:\n{chunk_text[:600]}\n\n"
        "Criterios:\n"
        "0 = el fragmento no aporta nada a la pregunta\n"
        "1 = menciona el tema pero no contesta\n"
        "2 = contiene información que responde directa o sustancialmente a la pregunta\n\n"
        "Responde SOLO con el dígito 0, 1 o 2."
    )
    try:
        resp = client.chat.completions.create(
            model=LLM_JUDGE_MODEL,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            temperature=0, max_tokens=2,
        )
        raw = resp.choices[0].message.content or ""
    except Exception:
        return -1
    ans = raw.strip()
    if JUDGE_DEBUG and _judge_debug_remaining > 0:
        print(f"  judge raw={raw!r}")
        _judge_debug_remaining -= 1
    for ch in ans:
        if ch in "012":
            return int(ch)
    return -1


def ndcg_at_k(rels, k):
    dcg = sum(r / np.log2(i + 2) for i, r in enumerate(rels[:k]))
    ideal = sorted(rels, reverse=True)[:k]
    idcg = sum(r / np.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def mrr(rels):
    for i, r in enumerate(rels):
        if r >= 1:
            return 1.0 / (i + 1)
    return 0.0


def average_precision(rels):
    rel_count, psum = 0, 0.0
    for i, r in enumerate(rels):
        if r >= 1:
            rel_count += 1
            psum += rel_count / (i + 1)
    return psum / rel_count if rel_count else 0.0


def _clean_rels(rels):
    return [max(r, 0) for r in rels]


def eval_variant(name, retrieve_fn, queries, k_list=EVAL_K_LIST, judge_fn=None):
    judge_fn = judge_fn or judge_relevance
    k_max = max(k_list)
    judgments = []
    n_errors = 0
    for q in queries:
        hits = retrieve_fn(q, k_max)
        rels = []
        for h in hits:
            txt = strip_doc_prefix(h["text"], EMBED_BACKEND)
            cid = h["meta"]["chunk_id"]
            try:
                r = judge_fn(q, txt, cid)
            except TypeError:
                r = judge_fn(q, txt)
            if r == -1:
                n_errors += 1
            rels.append(r)
        clean = _clean_rels(rels)
        entry = {"query": q, "relevances": rels, "clean_relevances": clean}
        for k in k_list:
            r_k = clean[:k]
            entry[f"precision@{k}"] = sum(1 for r in r_k if r >= 1) / k
            entry[f"ndcg@{k}"] = ndcg_at_k(r_k, k)
        entry["mrr"] = mrr(clean)
        entry["ap"] = average_precision(clean)
        judgments.append(entry)
    if n_errors:
        print(f"  ⚠️  {name}: {n_errors} juicios no parseables")
    summary = {"variant": name}
    for k in k_list:
        summary[f"Precision@{k}"] = np.mean([j[f"precision@{k}"] for j in judgments])
        summary[f"NDCG@{k}"] = np.mean([j[f"ndcg@{k}"] for j in judgments])
    summary["MRR"] = np.mean([j["mrr"] for j in judgments])
    summary["MAP"] = np.mean([j["ap"] for j in judgments])
    return summary, judgments


# Sanity clasificador
print("🧪 Clasificación benchmark (21 queries):")
correct = sum(
    1 for q in BENCHMARK_QUERIES
    if classify_query(q) == ("factual" if q in BENCHMARK_QUERIES_FACTUAL else "transversal")
)
print(f"  Aciertos: {correct}/{len(BENCHMARK_QUERIES)}")
print("✅ Eval listo")


🧪 Clasificación benchmark (21 queries):
  Aciertos: 21/21
✅ Eval listo


## 3. D1 — Referencias (`baseline` + `graph_router`)

Solo se ejecuta si `RUN_D1=True`. El desglose factual/transversal usa `RUN_D1_SPLIT` (por defecto **off** — se deriva de `router_d` sin juicios extra).


In [5]:
def _mean_ndcg_from_details(details, queries_subset, k=5):
    idx = [i for i, q in enumerate(BENCHMARK_QUERIES) if q in queries_subset]
    if not idx:
        return float("nan")
    return float(np.mean([details[i][f"ndcg@{k}"] for i in idx]))


if RUN_D1:
    print("📏 Evaluando baseline (21)...")
    base_s, base_d = eval_variant("baseline", retrieve_baseline_k, BENCHMARK_QUERIES)
    print("📏 Evaluando graph_router (21)...")
    router_s, router_d = eval_variant("graph_router", retrieve_router_k, BENCHMARK_QUERIES)
    df_ref = pd.DataFrame([base_s, router_s]).set_index("variant")
    print("\n🏆 Global (21 queries):")
    display(df_ref.sort_values("NDCG@5", ascending=False))

    if RUN_D1_SPLIT:
        print("\n📏 Factual (12) — eval completa...")
        base_f_s, base_f_d = eval_variant("baseline", retrieve_baseline_k, BENCHMARK_QUERIES_FACTUAL)
        router_f_s, router_f_d = eval_variant("graph_router", retrieve_router_k, BENCHMARK_QUERIES_FACTUAL)
        print("📏 Transversal (9)...")
        base_t_s, base_t_d = eval_variant("baseline", retrieve_baseline_k, BENCHMARK_QUERIES_TRANSVERSAL)
        router_t_s, router_t_d = eval_variant("graph_router", retrieve_router_k, BENCHMARK_QUERIES_TRANSVERSAL)
    else:
        print("\n📊 Desglose factual/transversal (derivado de D1, sin juicios extra):")
        rows = []
        for label, subset in [("factual", BENCHMARK_QUERIES_FACTUAL), ("transversal", BENCHMARK_QUERIES_TRANSVERSAL)]:
            rows.append({"split": label, "variant": "baseline", "NDCG@5": _mean_ndcg_from_details(base_d, subset)})
            rows.append({"split": label, "variant": "graph_router", "NDCG@5": _mean_ndcg_from_details(router_d, subset)})
        display(pd.DataFrame(rows).pivot(index="split", columns="variant", values="NDCG@5"))
        router_f_s = {"NDCG@5": _mean_ndcg_from_details(router_d, BENCHMARK_QUERIES_FACTUAL)}
        base_f_s = {"NDCG@5": _mean_ndcg_from_details(base_d, BENCHMARK_QUERIES_FACTUAL)}
        router_t_s = {"NDCG@5": _mean_ndcg_from_details(router_d, BENCHMARK_QUERIES_TRANSVERSAL)}
        base_t_s = {"NDCG@5": _mean_ndcg_from_details(base_d, BENCHMARK_QUERIES_TRANSVERSAL)}
        print(f"  Δ transversal: {router_t_s['NDCG@5'] - base_t_s['NDCG@5']:+.3f}")
        print(f"  Δ factual:     {router_f_s['NDCG@5'] - base_f_s['NDCG@5']:+.3f}")
else:
    print("⏭️  D1 saltado (RUN_D1=False) — necesario para D5")


📏 Evaluando baseline (21)...
📏 Evaluando graph_router (21)...

🏆 Global (21 queries):


,Precision@5,NDCG@5,Precision@10,NDCG@10,MRR,MAP
variant,,,,,,
graph_router,0.771429,0.953055,0.671429,0.914893,0.952381,0.861200
baseline,0.752381,0.939738,0.652381,0.918214,0.964286,0.856597



📊 Desglose factual/transversal (derivado de D1, sin juicios extra):


variant,baseline,graph_router
split,,
factual,0.985659,0.988409
transversal,0.878509,0.905916


  Δ transversal: +0.027
  Δ factual:     +0.003


## 4. Utilidades reranker (+ Sesión A opcional)

Define funciones de rerank. **Eval solo si `RUN_SESSION_A=True`** (descartado; n=20 archivado en `ARCHIVED_NDCG5`).


In [6]:
from sentence_transformers import CrossEncoder

_rerankers: dict[str, CrossEncoder] = {}


def get_reranker(model_id: str = RERANKER_MODEL) -> CrossEncoder:
    if model_id not in _rerankers:
        print(f"Cargando reranker: {model_id}...")
        _rerankers[model_id] = CrossEncoder(model_id, max_length=512)
    return _rerankers[model_id]


def rerank_chunks(query, chunks, top_n, model_id: str = RERANKER_MODEL):
    if not chunks:
        return []
    reranker = get_reranker(model_id)
    pairs = [(query, strip_doc_prefix(c["text"], EMBED_BACKEND)[:1500]) for c in chunks]
    scores = reranker.predict(pairs, batch_size=RERANK_BATCH_SIZE, show_progress_bar=False)
    ranked = sorted(zip(chunks, scores), key=lambda x: -float(x[1]))
    out = []
    for c, sc in ranked[:top_n]:
        c2 = dict(c)
        c2["rerank_score"] = float(sc)
        out.append(c2)
    return out


def retrieve_with_rerank(
    base_fn: Callable,
    query,
    top_n=TOP_N,
    model_id: str = RERANKER_MODEL,
    n_candidates: int | None = None,
):
    n = RERANK_CANDIDATES if n_candidates is None else n_candidates
    cands = base_fn(query, n)
    return rerank_chunks(query, cands, top_n, model_id=model_id)


def make_router_rerank_k(n_candidates: int):
    def fn(q, k):
        return retrieve_with_rerank(retrieve_router, q, k, n_candidates=n_candidates)
    return fn


def retrieve_baseline_rerank_k(q, k):
    return retrieve_with_rerank(retrieve_baseline, q, k)


def retrieve_router_rerank_k(q, k):
    return retrieve_with_rerank(retrieve_router, q, k)


br_s, br_d, rr_s, rr_d = None, None, None, None
if RUN_SESSION_A:
    print("📏 Evaluando baseline_rerank...")
    br_s, br_d = eval_variant("baseline_rerank", retrieve_baseline_rerank_k, BENCHMARK_QUERIES)
    print("📏 Evaluando graph_router_rerank...")
    rr_s, rr_d = eval_variant("graph_router_rerank", retrieve_router_rerank_k, BENCHMARK_QUERIES)
    df_rerank_a = pd.DataFrame([base_s, router_s, br_s, rr_s]).set_index("variant")
    print("\n🏆 Sesión A:")
    display(df_rerank_a.sort_values("NDCG@5", ascending=False))
else:
    print(f"⏭️  Sesión A saltada — graph_router_rerank archivado: NDCG@5={ARCHIVED_NDCG5.get('graph_router_rerank', '?')}")


⏭️  Sesión A saltada — graph_router_rerank archivado: NDCG@5=0.888


## 5. D2 — Reranker condicional (opcional, `RUN_D2=False` por defecto)

Descartado en Sesión D (NDCG@5 0.918 vs router 0.944).


In [ ]:
def retrieval_confidence(chunks):
    """Devuelve (top1_score, margin top1-top5)."""
    if not chunks:
        return 0.0, 0.0
    scores = [c.get("score", 0.0) for c in chunks[:5]]
    top1 = scores[0]
    top5 = scores[4] if len(scores) >= 5 else scores[-1]
    return top1, top1 - top5


def should_rerank(chunks, score_thr=RERANK_SCORE_THRESHOLD, margin_thr=RERANK_MARGIN_THRESHOLD) -> bool:
    top1, margin = retrieval_confidence(chunks)
    return top1 < score_thr or margin < margin_thr


def retrieve_with_conditional_rerank(
    base_fn: Callable,
    query,
    top_n=TOP_N,
    model_id: str = RERANKER_MODEL,
    score_thr=RERANK_SCORE_THRESHOLD,
    margin_thr=RERANK_MARGIN_THRESHOLD,
):
    cands = base_fn(query, RERANK_CANDIDATES)
    if should_rerank(cands, score_thr, margin_thr):
        out = rerank_chunks(query, cands, top_n, model_id=model_id)
        for c in out:
            c["rerank_applied"] = True
        return out
    out = [dict(c, rerank_applied=False) for c in cands[:top_n]]
    return out


def retrieve_router_cond_rerank_k(q, k):
    return retrieve_with_conditional_rerank(retrieve_router, q, k)


def retrieve_baseline_cond_rerank_k(q, k):
    return retrieve_with_conditional_rerank(retrieve_baseline, q, k)


rcr_s, rcr_d, bcr_s, bcr_d = None, None, None, None
if RUN_D2:
    _n_trigger = sum(1 for q in BENCHMARK_QUERIES if should_rerank(retrieve_router(q, RERANK_CANDIDATES)))
    print(f"🔎 Rerank condicional activo en {_n_trigger}/{len(BENCHMARK_QUERIES)} queries")
    print("📏 Evaluando router_cond_rerank...")
    rcr_s, rcr_d = eval_variant("router_cond_rerank", retrieve_router_cond_rerank_k, BENCHMARK_QUERIES)
    print("📏 Evaluando baseline_cond_rerank...")
    bcr_s, bcr_d = eval_variant("baseline_cond_rerank", retrieve_baseline_cond_rerank_k, BENCHMARK_QUERIES)
    rows = [base_s, router_s]
    if rr_s:
        rows.append(rr_s)
    rows.extend([rcr_s, bcr_s])
    display(pd.DataFrame(rows).set_index("variant").sort_values("NDCG@5", ascending=False))
else:
    print(f"⏭️  D2 saltado — router_cond_rerank archivado: NDCG@5={ARCHIVED_NDCG5.get('router_cond_rerank', '?')}")


## 6. D3 — Query decomposition (opcional, `RUN_D3=False`)

Funciones + diagnóstico de splits. Eval solo si `RUN_D3=True`.


In [ ]:
_DECOMPOSE_SYSTEM = (
    "Descompones preguntas sobre actas de obra ferroviaria en sub-preguntas INDEPENDIENTES.\n"
    "REGLAS OBLIGATORIAS:\n"
    "1. Si la pregunta une DOS temas con 'y' / 'además' → genera 2 sub-queries (una por tema).\n"
    "2. Si pide 'Compara A con B' → una sub-query por cada lado de la comparación.\n"
    "3. Si es un solo tema → devuelve SOLO esa pregunta en el array.\n"
    "4. Cada sub-query debe poder responderse de forma autónoma.\n"
    f"Responde SOLO JSON: {{\"sub_queries\": [\"...\"]}} (máximo {MAX_SUB_QUERIES})."
)

_DECOMPOSE_FEW_SHOT = [
    ("¿Qué se decidió sobre el talud?", '{"sub_queries": ["¿Qué se decidió sobre el talud?"]}'),
    (
        "¿Qué se decidió sobre el talud y qué acciones quedan pendientes?",
        '{"sub_queries": ["¿Qué se decidió sobre el talud?", "¿Qué acciones quedan pendientes sobre el talud?"]}',
    ),
    (
        "Compara las incidencias de luminarias con las de instalaciones eléctricas en las actas",
        '{"sub_queries": ["¿Qué incidencias de luminarias aparecen en las actas?", "¿Qué incidencias de instalaciones eléctricas aparecen en las actas?"]}',
    ),
    (
        "¿Qué documentación debe aportar la UTE y cuál es el estado de las instalaciones de megafonía?",
        '{"sub_queries": ["¿Qué documentación debe aportar la UTE?", "¿Cuál es el estado de las instalaciones de megafonía?"]}',
    ),
    (
        "Lista las solicitudes a la constructora y agrupa las incidencias por tipo constructivo",
        '{"sub_queries": ["Lista todas las solicitudes realizadas a la constructora", "Agrupa las incidencias por tipo constructivo en las actas"]}',
    ),
]

_decompose_cache: dict[str, list[str]] = {}


def _heuristic_decompose(query: str) -> list[str] | None:
    q = query.strip()
    ql = q.lower()
    # Compara A con B
    m = re.search(r"compara\\s+(.+?)\\s+con\\s+(.+?)(?:\\s+en las actas)?[?.!]?$", ql, re.I)
    if m:
        a, b = m.group(1).strip(" ?."), b.strip(" ?.")
        return [
            f"¿Qué incidencias o información sobre {a} aparecen en las actas?",
            f"¿Qué incidencias o información sobre {b} aparecen en las actas?",
        ]
    # Split por " y " si ambas partes tienen sustancia
    if " y " in ql:
        parts = re.split(r"\\s+y\\s+", q, maxsplit=1, flags=re.I)
        if len(parts) == 2 and all(len(p.strip()) > 15 for p in parts):
            left, right = parts[0].strip(), parts[1].strip()
            if not left.endswith("?"):
                left = left if left.startswith("¿") else f"¿{left}?"
            if not right.endswith("?"):
                right = right if right.startswith("¿") else f"¿{right}?"
            return [left, right]
    return None


def decompose_query(query: str, use_heuristic: bool = True) -> list[str]:
    if query in _decompose_cache:
        return _decompose_cache[query]
    msgs = [{"role": "system", "content": _DECOMPOSE_SYSTEM}]
    for q, ans in _DECOMPOSE_FEW_SHOT:
        msgs.append({"role": "user", "content": q})
        msgs.append({"role": "assistant", "content": ans})
    msgs.append({"role": "user", "content": query})
    subs = None
    try:
        resp = client.chat.completions.create(
            model=DECOMPOSE_MODEL, messages=msgs, temperature=0, max_tokens=250,
        )
        raw = (resp.choices[0].message.content or "").strip()
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        data = json.loads(m.group(0)) if m else {}
        subs = [s.strip() for s in data.get("sub_queries", []) if s and str(s).strip()][:MAX_SUB_QUERIES]
    except Exception:
        subs = None
    if not subs or (len(subs) == 1 and subs[0].strip().lower() == query.strip().lower()):
        if use_heuristic:
            h = _heuristic_decompose(query)
            if h:
                subs = h
    if not subs:
        subs = [query]
    _decompose_cache[query] = subs
    return subs


def merge_chunk_lists(chunk_lists, top_n):
    merged = {}
    for chunks in chunk_lists:
        for c in chunks:
            cid = c["meta"]["chunk_id"]
            if cid not in merged:
                merged[cid] = dict(c)
            else:
                merged[cid]["score"] = merged[cid].get("score", 0) + c.get("score", 0)
    return sorted(merged.values(), key=lambda x: x["score"], reverse=True)[:top_n]


def retrieve_decomposed(base_fn: Callable, query, top_n=TOP_N):
    subs = decompose_query(query)
    if len(subs) == 1:
        return base_fn(query, top_n)
    lists = [base_fn(sq, top_n) for sq in subs]
    return merge_chunk_lists(lists, top_n)


def retrieve_decompose_baseline_k(q, k):
    return retrieve_decomposed(retrieve_baseline, q, k)


def retrieve_decompose_router_k(q, k):
    return retrieve_decomposed(retrieve_router, q, k)


if RUN_D3:
    print("🔎 Diagnóstico decomposition (8 compuestas):")
    diag_rows = []
    for q in BENCHMARK_QUERIES_COMPOSITE_ALL:
        subs = decompose_query(q)
        diag_rows.append({
            "query": q[:55],
            "n_subs": len(subs),
            "split_ok": len(subs) > 1,
            "sub_queries": " | ".join(s[:40] for s in subs),
        })
    df_diag = pd.DataFrame(diag_rows)
    display(df_diag)
    print(f"  Splits correctos (>1 sub-query): {df_diag['split_ok'].sum()}/{len(df_diag)}")
else:
    print("⏭️  D3 funciones cargadas; diagnóstico saltado (RUN_D3=False)")


## 7. D3 — Eval decomposition + variante rerank por sub-query

- `decompose_router` / `decompose_baseline` en 8 compuestas
- `decompose_router_rerank_v2`: rerankea **cada sub-query con su propio texto** y fusiona después (arregla desalineación de Sesión A)


In [ ]:
def retrieve_decompose_router_rerank_v2_k(q, k):
    subs = decompose_query(q)
    if len(subs) == 1:
        return retrieve_with_conditional_rerank(retrieve_router, q, k)
    lists = [retrieve_with_rerank(retrieve_router, sq, k) for sq in subs]
    return merge_chunk_lists(lists, k)


cb_s, cr_s, dcb_s, dcr_s, dcrr2_s, df_comp = None, None, None, None, None, None
if RUN_D3:
    print("📏 Eval composite (8)...")
    cb_s, cb_d = eval_variant("baseline", retrieve_baseline_k, BENCHMARK_QUERIES_COMPOSITE_ALL)
    cr_s, cr_d = eval_variant("graph_router", retrieve_router_k, BENCHMARK_QUERIES_COMPOSITE_ALL)
    dcb_s, dcb_d = eval_variant("decompose_baseline", retrieve_decompose_baseline_k, BENCHMARK_QUERIES_COMPOSITE_ALL)
    dcr_s, dcr_d = eval_variant("decompose_router", retrieve_decompose_router_k, BENCHMARK_QUERIES_COMPOSITE_ALL)
    dcrr2_s, dcrr2_d = eval_variant("decompose_router_rerank_v2", retrieve_decompose_router_rerank_v2_k, BENCHMARK_QUERIES_COMPOSITE_ALL)
    df_comp = pd.DataFrame([cb_s, cr_s, dcb_s, dcr_s, dcrr2_s]).set_index("variant")
    display(df_comp.sort_values("NDCG@5", ascending=False))
else:
    print(f"⏭️  D3 eval saltada — decompose_router archivado: NDCG@5={ARCHIVED_NDCG5.get('decompose_router', '?')} (8 compuestas)")


## 8. D4 — Reranker m3 (opcional, `RUN_D4=False`)


In [ ]:
def retrieve_router_cond_rerank_m3_k(q, k):
    return retrieve_with_conditional_rerank(retrieve_router, q, k, model_id=RERANKER_MODEL_M3)


def retrieve_router_rerank_m3_k(q, k):
    return retrieve_with_rerank(retrieve_router, q, k, model_id=RERANKER_MODEL_M3)


rcm3_s, rrm3_s = None, None
if RUN_D4:
    rcm3_s, _ = eval_variant("router_cond_rerank_m3", retrieve_router_cond_rerank_m3_k, BENCHMARK_QUERIES)
    rrm3_s, _ = eval_variant("graph_router_rerank_m3", retrieve_router_rerank_m3_k, BENCHMARK_QUERIES)
    display(pd.DataFrame([router_s, rcr_s, rcm3_s, rr_s, rrm3_s]).set_index("variant").sort_values("NDCG@5", ascending=False))
else:
    print(f"⏭️  D4 saltada — m3 archivado: cond={ARCHIVED_NDCG5.get('router_cond_rerank_m3')}, always={ARCHIVED_NDCG5.get('graph_router_rerank_m3')}")


## 9. D5 — Sweep pool de candidatos al reranker

**Hipótesis:** con `RERANK_CANDIDATES=20` el reranker no puede rescatar chunks en posiciones 21–100 del RRF (literatura recomienda **50–100**).

1. **D5.1** Diagnóstico Recall@K del RRF (`graph_router`, sin reranker)
2. **D5.2** Sweep `router_rerank_n{50,100}` (n=20 archivado en `ARCHIVED_NDCG5`)
3. **D5.3** Δ por query + desglose factual / transversal
4. **D5.4** (opcional) `bge-reranker-v2-m3` con el mejor pool si supera al router


In [7]:
if not RUN_D5:
    print("⏭️  D5 desactivado (RUN_D5=False)")
elif not RUN_D1:
    raise RuntimeError("D5 requiere RUN_D1=True (baseline + router)")
else:
    pass  # continúa celdas D5.1–D5.4


In [ ]:
# ─── Cache de juicios (reutiliza entre variantes D5) ─────────────────
_judge_cache: dict[tuple[str, str], int] = {}


def judge_relevance_cached(query: str, chunk_text: str, chunk_id: str) -> int:
    key = (query, chunk_id)
    if key not in _judge_cache:
        _judge_cache[key] = judge_relevance(query, chunk_text)
    return _judge_cache[key]


def diagnose_rrf_recall(retrieve_fn, queries, k_max=D5_RECALL_K):
    """Posición del primer chunk relevante (judge≥1) en el ranking RRF."""
    rows = []
    for q in queries:
        hits = retrieve_fn(q, k_max)
        first_rel = None
        for i, h in enumerate(hits):
            txt = strip_doc_prefix(h["text"], EMBED_BACKEND)
            cid = h["meta"]["chunk_id"]
            r = judge_relevance_cached(q, txt, cid)
            if r >= 1 and first_rel is None:
                first_rel = i + 1
        bucket = ">100" if first_rel is None else (
            "1-20" if first_rel <= 20 else "21-50" if first_rel <= 50 else "51-100"
        )
        rows.append({
            "query": q[:50],
            "first_rel_rank": first_rel,
            "bucket": bucket,
            "split": "factual" if q in BENCHMARK_QUERIES_FACTUAL else "transversal",
        })
    return pd.DataFrame(rows)


d5_summaries, d5_details, df_recall, best_n, d5_m3_s, d5_m3_d = {}, {}, None, None, None, None

if RUN_D5:
    print(f"🔎 D5.1 — Recall RRF (top-{D5_RECALL_K})...")
    df_recall = diagnose_rrf_recall(retrieve_router_k, BENCHMARK_QUERIES)
    display(df_recall.sort_values("first_rel_rank", na_position="last"))
    n = len(df_recall)
    for label, cond in [
        ("top-20", df_recall["first_rel_rank"].le(20)),
        ("21-50", df_recall["first_rel_rank"].between(21, 50)),
        ("51-100", df_recall["first_rel_rank"].between(51, 100)),
        ("no relevante en 100", df_recall["first_rel_rank"].isna()),
    ]:
        print(f"  {label}: {cond.sum()}/{n} ({100*cond.sum()/n:.0f}%)")
    pct_21_50 = df_recall["first_rel_rank"].between(21, 50).sum() / n
    if pct_21_50 >= 0.15:
        print(f"\n✅ H1 plausible: {pct_21_50:.0%} en pos 21–50")
    else:
        print(f"\n⚠️  H1 débil: {pct_21_50:.0%} en pos 21–50")


🔎 D5.1 — Recall RRF (top-100)...


In [ ]:
if RUN_D5:
    for n_cand in D5_CANDIDATE_SWEEP:
        name = f"router_rerank_n{n_cand}"
        print(f"📏 D5.2 — {name} (pool={n_cand})...")
        s, d = eval_variant(name, make_router_rerank_k(n_cand), BENCHMARK_QUERIES, judge_fn=judge_relevance_cached)
        d5_summaries[n_cand] = s
        d5_details[n_cand] = d

    rows_d5 = [{"variant": "graph_router", **{k: v for k, v in router_s.items() if k != "variant"}}]
    for n_cand in D5_CANDIDATE_SWEEP:
        rows_d5.append(d5_summaries[n_cand])
    if 20 not in D5_CANDIDATE_SWEEP and "graph_router_rerank" in ARCHIVED_NDCG5:
        rows_d5.append({"variant": "router_rerank_n20_archived", "NDCG@5": ARCHIVED_NDCG5["graph_router_rerank"]})
    df_d5_sweep = pd.DataFrame(rows_d5).set_index("variant")
    print("\n🏆 D5.2 — sweep pool:")
    display(df_d5_sweep.sort_values("NDCG@5", ascending=False))

    best_n = max(D5_CANDIDATE_SWEEP, key=lambda n: d5_summaries[n]["NDCG@5"])
    n20_ref = ARCHIVED_NDCG5.get("graph_router_rerank") if 20 not in D5_CANDIDATE_SWEEP else d5_summaries.get(20, {}).get("NDCG@5")
    print(f"\n  Mejor pool: n={best_n} ({d5_summaries[best_n]['NDCG@5']:.3f})")
    print(f"  vs router: {d5_summaries[best_n]['NDCG@5'] - router_s['NDCG@5']:+.3f}")
    if n20_ref is not None:
        print(f"  vs n=20 (archivado): {d5_summaries[best_n]['NDCG@5'] - n20_ref:+.3f}")


In [ ]:
if RUN_D5:
    K_REF = 5
    delta_d5 = []
    for i, q in enumerate(BENCHMARK_QUERIES):
        row = {
            "query": q[:48],
            "split": "F" if q in BENCHMARK_QUERIES_FACTUAL else "T",
            "router": router_d[i][f"ndcg@{K_REF}"],
            "recall_rank": df_recall.iloc[i]["first_rel_rank"],
        }
        for n_cand in D5_CANDIDATE_SWEEP:
            row[f"n{n_cand}"] = d5_details[n_cand][i][f"ndcg@{K_REF}"]
            row[f"Δn{n_cand}"] = row[f"n{n_cand}"] - row["router"]
        delta_d5.append(row)
    print("📋 D5.3 — Δ por query:")
    display(pd.DataFrame(delta_d5).sort_values(f"Δn{best_n}", ascending=False))

    print("\n📊 D5.3 — desglose por split (derivado, sin juicios extra):")
    for split_name, subset in [("factual", BENCHMARK_QUERIES_FACTUAL), ("transversal", BENCHMARK_QUERIES_TRANSVERSAL)]:
        sub_rows = [{"pool": "router", "NDCG@5": _mean_ndcg_from_details(router_d, subset)}]
        for n_cand in D5_CANDIDATE_SWEEP:
            sub_rows.append({"pool": n_cand, "NDCG@5": _mean_ndcg_from_details(d5_details[n_cand], subset)})
        print(f"  {split_name}:")
        display(pd.DataFrame(sub_rows).set_index("pool").sort_values("NDCG@5", ascending=False))


In [ ]:
if RUN_D5 and D5_RUN_M3_IF_IMPROVES and d5_summaries[best_n]["NDCG@5"] > router_s["NDCG@5"]:
    def retrieve_router_rerank_best_m3_k(q, k):
        return retrieve_with_rerank(
            retrieve_router, q, k, model_id=RERANKER_MODEL_M3, n_candidates=best_n
        )
    print(f"📏 D5.4 — router_rerank_n{best_n}_m3 (pool={best_n})...")
    d5_m3_s, d5_m3_d = eval_variant(
        f"router_rerank_n{best_n}_m3", retrieve_router_rerank_best_m3_k, BENCHMARK_QUERIES,
        judge_fn=judge_relevance_cached,
    )
    df_d5_m3 = pd.DataFrame([router_s, d5_summaries[best_n], d5_m3_s]).set_index("variant")
    print(f"\n🏆 D5.4 — base n={best_n} vs m3:")
    display(df_d5_m3.sort_values("NDCG@5", ascending=False))
elif RUN_D5:
    why = "D5_RUN_M3_IF_IMPROVES=False" if not D5_RUN_M3_IF_IMPROVES else f"n={best_n} no supera router"
    print(f"⏭️  D5.4 saltada ({why})")


## 10. Comparativa final (D1–D5)

Tabla unificada + conclusiones automáticas.


In [ ]:
# Esta ejecución
rows_run = []
if RUN_D1:
    rows_run.extend([base_s, router_s])
for n_cand in D5_CANDIDATE_SWEEP:
    if n_cand in d5_summaries:
        rows_run.append(d5_summaries[n_cand])
if d5_m3_s is not None:
    rows_run.append(d5_m3_s)
if rows_run:
    df_final = pd.DataFrame(rows_run).set_index("variant")
    print("🏆 Esta ejecución (21 queries):")
    display(df_final.sort_values("NDCG@5", ascending=False))

# Histórico archivado (sin re-ejecutar)
if ARCHIVED_NDCG5:
    print("\n📁 Archivado (Sesiones A/D previas):")
    display(pd.DataFrame([{"variant": k, "NDCG@5": v} for k, v in ARCHIVED_NDCG5.items()]).set_index("variant"))

if RUN_D1 and RUN_D5 and best_n in d5_summaries:
    print("\n📌 Conclusiones D5:")
    print(f"  • Router vs baseline: {router_s['NDCG@5'] - base_s['NDCG@5']:+.3f}")
    print(f"  • Mejor pool n={best_n} vs router: {d5_summaries[best_n]['NDCG@5'] - router_s['NDCG@5']:+.3f}")
    n20 = ARCHIVED_NDCG5.get("graph_router_rerank")
    if n20 is not None:
        print(f"  • n={best_n} vs n=20 archivado: {d5_summaries[best_n]['NDCG@5'] - n20:+.3f}")
    if d5_summaries[best_n]["NDCG@5"] > router_s["NDCG@5"] + 0.01:
        print(f"  → Integrar reranker pool={best_n}")
    elif d5_summaries[best_n]["NDCG@5"] > router_s["NDCG@5"]:
        print("  → Ganancia marginal; valorar solo transversales")
    else:
        print("  → Descartar reranker")
